In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

TMA_Core="A" # Should be A,B,C 

# #Load files on laptop
# expr= pd.read_csv(fr"C:\Users\evanj\OneDrive\Documents\Shapiro Data Files\Laptop Cutdown RNA Files\RNAexpression_TMA{TMA_Core}_transposed.csv", index_col=0)
# #expr=expr_orig.transpose()
# metadata = pd.read_csv(fr"C:\Users\evanj\OneDrive\Documents\Shapiro Data Files\Laptop Cutdown RNA Files\RNA_TMA{TMA_Core}_meta_transposed.csv", index_col=0)
# umap = pd.read_csv(fr"C:\Users\evanj\OneDrive\Documents\Shapiro Data Files\Laptop Cutdown RNA Files\RNA_TMA{TMA_Core}_umap_transposed.csv", index_col=0)

#Load files on desktop
expr= pd.read_csv(fr"C:\Users\ejohns\Documents\Shapiro Data Files\CutdownRNAFiles\RNAexpression_TMA{TMA_Core}_transposed.csv", index_col=0)
#expr=expr_orig.transpose()
metadata = pd.read_csv(fr"C:\Users\ejohns\Documents\Shapiro Data Files\CutdownRNAFiles\RNA_TMA{TMA_Core}_meta_transposed.csv", index_col=0)
umap = pd.read_csv(fr"C:\Users\ejohns\Documents\Shapiro Data Files\CutdownRNAFiles\RNA_TMA{TMA_Core}_umap_transposed.csv", index_col=0)

# Extra - MetaData File
extra_metadata=pd.read_csv(fr"C:\Users\ejohns\Documents\Shapiro Data Files\CutdownRNAFiles\Run_TMA{TMA_Core}_metadata_file.csv", index_col=0)

# Create AnnData object
adata_rna = sc.AnnData(X=expr.values)

# Assign metadata
adata_rna.obs = metadata
adata_rna.var_names = expr.columns
adata_rna.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
adata_rna.obsm["spatial"] = metadata[['x_pixel', 'y_pixel']].values  # adjust if needed
# x_pixel, y_pixel, x_norm	y_norm	x_micron	y_micron x_cent	y_cent

adata_rna.obsm["X_umap"] = umap.values

C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import annd

In [11]:
def find_matching_global_coords(extra_metadata, cell_num, FOV_num):
    import numpy as np
    import pandas as pd
    
    # Ensure the index is reset so FOV becomes a column (if needed)
    filtered_row = extra_metadata.reset_index()
    filtered_row = filtered_row[(filtered_row['fov'] == FOV_num) & (filtered_row['cell_ID'] == cell_num)]

    if filtered_row.empty:
        print(f"No match found for FOV={FOV_num} and cell_ID={cell_num}.")
        return None

    global_x = filtered_row['CenterX_global_px'].values[0] * 0.168 * 0.001
    global_y = filtered_row['CenterY_global_px'].values[0] * 0.168 * 0.001

    return {
        "Global X": global_x,
        "Global Y": global_y
    }
results=find_matching_global_coords(extra_metadata, 8, 1)
display(results)

{'Global X': 45.91026400000001, 'Global Y': 10.246440000000003}

In [12]:
### Function to add global coordinates to the metadata from extra metadata. 

import pandas as pd
import numpy as np
import re

def find_matching_global_coords(extra_metadata, cell_num, FOV_num):
    filtered_row = extra_metadata.reset_index()
    filtered_row = filtered_row[
        (filtered_row['fov'] == FOV_num) & (filtered_row['cell_ID'] == cell_num)
    ]

    if filtered_row.empty:
        print(f"No match found for FOV={FOV_num} and cell_ID={cell_num}.")
        return None

    global_x = filtered_row['CenterX_global_px'].values[0] * 0.168 * 0.001
    global_y = filtered_row['CenterY_global_px'].values[0] * 0.168 * 0.001

    return {
        "Global X": global_x,
        "Global Y": global_y
    }

def add_global_coords_to_metadata(metadata, extra_metadata):
    # Ensure the index is a column for processing
    metadata = metadata.copy()
    metadata['cell_id_str'] = metadata.index.astype(str)

    # Function to extract coordinates
    def extract_and_get_coords(cell_id_str):
        match = re.search(r'_Run\d+\.TMA[A-Za-z]_(\d+)_(\d+)', cell_id_str)
        if match:
            cell_num = int(match.group(1))
            FOV_num = int(match.group(2))
            coords = find_matching_global_coords(extra_metadata, cell_num, FOV_num)
            if coords:
                return pd.Series([coords["Global X"], coords["Global Y"]])
        return pd.Series([None, None])

    metadata[['global_x_coord', 'global_y_coord']] = metadata['cell_id_str'].apply(extract_and_get_coords)
    metadata.drop(columns=['cell_id_str'], inplace=True)

    return metadata



results = add_global_coords_to_metadata(metadata, extra_metadata)